# 03 - Entrenamiento de Modelos con MLflow

Entrenamos cuatro clasificadores para predecir `late_delivery_risk` y comparamos su desempeño usando MLflow para el tracking.

## Modelos

| Modelo | Tipo | Escalado | Notas |
|--------|------|----------|-------|
| **Logistic Regression** | Lineal | StandardScaler | Pipeline con `class_weight='balanced'` |
| **Random Forest** | Ensemble bagging | No | 300 árboles, `class_weight='balanced'` |
| **XGBoost** | Gradient boosting | No | `scale_pos_weight` automático |
| **LightGBM** | Gradient boosting | No | Rápido, `num_leaves=64` |

## Pipeline profesional

1. Carga de `X_train`, `y_train`, `X_test`, `y_test` desde `data/processed/`.
2. Entrenamiento de los 4 modelos como **nested runs** de un experimento MLflow.
3. Registro de métricas: `accuracy`, `precision`, `recall`, `F1`, `ROC-AUC`.
4. Generación de **matrices de confusión** y **feature importance** como artefactos.
5. Persistencia de cada modelo en `models/`.
6. Comparativa final + **selección del mejor modelo** según ROC-AUC.

Abre MLflow UI en otra terminal:

```bash
mlflow ui --backend-store-uri mlruns/
```

## 0. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import configurar_logger
from src.train_model import (
    entrenar_logistic_regression,
    entrenar_random_forest,
    entrenar_xgboost,
    entrenar_lightgbm,
    entrenar_todos_los_modelos,
    comparar_modelos,
    seleccionar_mejor_modelo,
    PARAMS_LOGISTIC,
    PARAMS_RANDOM_FOREST,
    PARAMS_XGBOOST,
    PARAMS_LIGHTGBM,
)
from src.features import TARGET

configurar_logger('training')
sns.set_theme(style='whitegrid', palette='viridis')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:,.4f}'.format)

## 1. Cargar los datasets de modelado

Producidos en `02_feature_engineering.ipynb`.

In [ ]:
ruta_train = Path('../data/processed/dataco_train.parquet')
ruta_test = Path('../data/processed/dataco_test.parquet')

if not ruta_train.exists() or not ruta_test.exists():
    raise FileNotFoundError(
        'No se encontraron los parquets de train/test.\n'
        'Ejecuta primero el notebook 02_feature_engineering.ipynb.'
    )

df_train = pd.read_parquet(ruta_train)
df_test = pd.read_parquet(ruta_test)

X_train = df_train.drop(columns=[TARGET])
y_train = df_train[TARGET].astype(int)
X_test = df_test.drop(columns=[TARGET])
y_test = df_test[TARGET].astype(int)

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'Tasa de retraso (train): {y_train.mean():.2%}')
print(f'Tasa de retraso (test):  {y_test.mean():.2%}')

## 2. Hiperparámetros por defecto

Cada modelo trae una configuración base sensata definida en `src/train_model.py`. Puedes sobrescribirla pasando un dict propio a cada `entrenar_*`.

In [ ]:
for nombre, params in [
    ('Logistic Regression', PARAMS_LOGISTIC),
    ('Random Forest', PARAMS_RANDOM_FOREST),
    ('XGBoost', PARAMS_XGBOOST),
    ('LightGBM', PARAMS_LIGHTGBM),
]:
    print(f'\n>>> {nombre}')
    for k, v in params.items():
        print(f'    {k}: {v}')

## 3. Entrenamiento de los 4 modelos

Usamos el orquestador `entrenar_todos_los_modelos`, que ejecuta cada entrenamiento como **nested run** dentro de un experimento padre llamado `comparativa_modelos` en MLflow.

In [ ]:
resultados = entrenar_todos_los_modelos(
    X_train, y_train, X_test, y_test,
    modelos=['logistic_regression', 'random_forest', 'xgboost', 'lightgbm'],
)

## 4. Comparativa de métricas

In [ ]:
df_comp = comparar_modelos(resultados)
df_comp

In [ ]:
metricas_plot = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
df_long = df_comp.melt(
    id_vars='modelo', value_vars=metricas_plot,
    var_name='metrica', value_name='valor',
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=df_long, x='metrica', y='valor', hue='modelo', ax=ax)
ax.set_title('Comparativa de modelos en test set')
ax.set_ylim(0, 1)
ax.set_ylabel('Valor de la métrica')
ax.legend(loc='lower right', bbox_to_anchor=(1.25, 0))
plt.tight_layout()
plt.show()

## 5. Matrices de confusión

In [ ]:
n_modelos = len(resultados)
fig, axes = plt.subplots(1, n_modelos, figsize=(4 * n_modelos, 4))
if n_modelos == 1:
    axes = [axes]

for ax, (nombre, r) in zip(axes, resultados.items()):
    sns.heatmap(
        r.matriz_confusion,
        annot=True, fmt='d', cmap='Blues', cbar=False,
        xticklabels=['A tiempo', 'Retraso'],
        yticklabels=['A tiempo', 'Retraso'],
        ax=ax,
    )
    ax.set_title(f"{nombre}\nROC-AUC={r.metricas['roc_auc']:.3f}")
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')

plt.tight_layout()
plt.show()

## 6. Feature Importance por modelo

Cada uno aporta una perspectiva distinta:
- **Logistic Regression**: coeficientes absolutos (lineal).
- **Random Forest / XGBoost / LightGBM**: importancia basada en *gain* o reducciones de impureza.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (nombre, r) in zip(axes.flat, resultados.items()):
    if r.feature_importance is None:
        ax.set_visible(False)
        continue
    top = r.feature_importance.head(15).iloc[::-1]
    top.plot(kind='barh', ax=ax, color='#4c8acc')
    ax.set_title(f'Top 15 features — {nombre}')
    ax.set_xlabel('Importancia')

plt.tight_layout()
plt.show()

### Features comunes entre los modelos top

In [ ]:
top_n = 15
sets_top = {
    nombre: set(r.feature_importance.head(top_n).index)
    for nombre, r in resultados.items()
    if r.feature_importance is not None
}
comunes = set.intersection(*sets_top.values()) if sets_top else set()
print(f'Features presentes en el TOP-{top_n} de TODOS los modelos:\n')
for f in sorted(comunes):
    print(f'  - {f}')

## 7. Curvas ROC

In [ ]:
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8, 6))
for nombre, r in resultados.items():
    proba = r.modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=f"{nombre} (AUC={r.metricas['roc_auc']:.3f})")

ax.plot([0, 1], [0, 1], '--', color='gray', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Curvas ROC — comparativa de modelos')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 8. Curvas Precision-Recall

Más informativa que ROC cuando las clases están desbalanceadas o cuando el costo de falsos negativos es alto (¡como en retrasos!).

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

fig, ax = plt.subplots(figsize=(8, 6))
for nombre, r in resultados.items():
    proba = r.modelo.predict_proba(X_test)[:, 1]
    p, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(rec, p, label=f"{nombre} (AP={ap:.3f})")

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Curvas Precision-Recall')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

## 9. Selección automática del mejor modelo

Criterio por defecto: **ROC-AUC** (puede cambiarse).

El ganador se guarda en `models/modelo_mejor.joblib` (alias) y queda disponible para la API de predicción.

In [ ]:
mejor = seleccionar_mejor_modelo(resultados, metrica='roc_auc')

print(f'Modelo ganador: {mejor.nombre}')
print(f'Run ID:         {mejor.run_id}')
print(f'Ruta del modelo: {mejor.ruta_modelo}')
print('\nMétricas:')
for k, v in mejor.metricas.items():
    print(f'  {k:12s}: {v:.4f}')

### Reporte detallado del ganador

In [ ]:
from sklearn.metrics import classification_report

pred_test = mejor.modelo.predict(X_test)
print(classification_report(y_test, pred_test, digits=4,
                            target_names=['A tiempo (0)', 'Retraso (1)']))

## 10. Probar la inferencia desde `src.predict`

In [ ]:
from src.predict import predecir_retraso, limpiar_cache_modelos

limpiar_cache_modelos()  # Asegura que se recargue el modelo recién guardado

muestra = X_test.iloc[:5]
resultado = predecir_retraso(muestra)

pd.DataFrame({
    'prob_retraso': resultado['probabilidad_retraso'],
    'prediccion': resultado['prediccion'],
    'nivel_riesgo': resultado['nivel_riesgo'],
    'real': y_test.iloc[:5].values,
})

## 11. Conclusiones y próximos pasos

- Los 4 modelos quedaron registrados en MLflow bajo el experimento configurado en `.env`.
- El ganador se sirve automáticamente desde la API (`/predicciones/retraso`) gracias al alias `modelo_mejor.joblib`.
- Artefactos en `reports/`: matrices de confusión, feature importance (CSV y PNG) y `comparativa_modelos.csv`.

### Siguientes pasos
1. **Tuning de hiperparámetros** del modelo ganador (Optuna o GridSearchCV).
2. **Análisis SHAP** para explicar predicciones individuales en producción.
3. **Calibración de probabilidades** (`CalibratedClassifierCV`).
4. **Monitoreo de drift** entre la distribución de train y la de producción.
5. Pasar al notebook `04_financial_impact.ipynb` para cuantificar el ROI del modelo.